[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C45_Privacy_Trustworthy_Course/04_unlearning/04_unlearning.ipynb)

# 04 · 机器遗忘（用 numpy 从零实现）

目标：把 **重训(金标准)**、**SISA 分片**、**影响函数近似遗忘**、**成员推断验证** 用 numpy 实现，并 `assert` 验证。

路线：训练 + 重训(参照) → 影响函数一步遗忘(对拍重训) → SISA 分片(只重训 1/S) → 成员推断验证「忘了没」 → ✏️ 练习 → 📖 答案 → 🧪 真实数据胶囊。

> 心智模型：**重训 = 金标准**（完美但贵）。影响函数 = 一步参数修正逼近重训（便宜但近似）。SISA = 只重训目标所在分片。**遗忘必须验证**（MIA 掉回随机水平 = 忘干净）。
> 用**强凸**的 L2 逻辑回归当靶子——此时 Hessian 可逆、影响函数准，能干净地对拍重训。

## 1 · 训练 + 重训（金标准参照）

先训一个带 L2 的逻辑回归（强凸 -> 有唯一最优解、Hessian 正定可逆）。
再实现「重训」——删掉某样本后从头训：这是遗忘正确性的**金标准**，一切方法都跟它比。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

LAMBDA = 0.5     # L2 正则强度（保证强凸）

def make_data(n=300, d=6, seed=0):
    g = np.random.default_rng(seed)
    w = g.standard_normal(d)
    X = g.standard_normal((n, d))
    y = (X @ w + 0.5*g.standard_normal(n) > 0).astype(float)
    return X, y

def sigmoid(z):
    return 1.0/(1.0+np.exp(-np.clip(z, -30, 30)))

def train_lr(X, y, lam=LAMBDA, iters=500, lr=0.3):
    '''L2 逻辑回归，梯度下降到收敛(强凸 -> 唯一最优)。
       lr 取 0.3 保证对本课用到的 lam(0.01~5.0) 都数值稳定：
       梯度下降在 lr*(数据曲率+lam) > 2 时会发散，lr=0.3 留足裕度。'''
    n, d = X.shape
    w = np.zeros(d)
    for _ in range(iters):
        p = sigmoid(X @ w)
        grad = X.T @ (p - y) / n + lam * w
        w -= lr * grad
    return w

X, y = make_data()
w_full = train_lr(X, y)                 # 在全量数据上训练

# 选一个要遗忘的样本 z（下标 0），重训 = 删掉它从头训
j = 0
X_minus = np.delete(X, j, axis=0); y_minus = np.delete(y, j)
w_retrain = train_lr(X_minus, y_minus)  # 金标准：从没见过 z 的模型
print('全量模型   w_full[:3]    =', w_full[:3].round(3))
print('重训模型   w_retrain[:3] =', w_retrain[:3].round(3))
print(f'删一个样本使参数变化 L2 = {np.linalg.norm(w_full - w_retrain):.4f}')
assert np.linalg.norm(w_full - w_retrain) > 0, '删样本应改变最优参数'
print('✅ 金标准就绪：重训 = 用「删掉 z 的数据」从头训，遗忘要逼近它')

## 2 · 影响函数：一步逼近重训（不重训）

影响函数：`θ_{-z} ≈ θ̂ + (1/n)·H⁻¹·∇L(z)`。撤销 z 当初施加的「拉力」，用 Hessian 逆按曲率缩放。

实现它，**对拍重训**：影响函数一步更新应比「啥也不做」更接近重训结果。

In [ ]:
def hessian_lr(w, X, y, lam=LAMBDA):
    '''L2 逻辑回归的 Hessian: (1/n) Σ p(1-p) x xᵀ + λI。强凸 -> 正定可逆。'''
    n, d = X.shape
    p = sigmoid(X @ w)
    W = p * (1 - p)                       # (n,)
    H = (X * W[:, None]).T @ X / n + lam * np.eye(d)
    return H

def grad_single(w, x, yi, lam=LAMBDA, n=1):
    '''单样本对总损失的梯度贡献(含其那份正则可忽略，用经验风险梯度)。'''
    p = sigmoid(x @ w)
    return (p - yi) * x                    # 单样本损失梯度

def unlearn_influence(w, X, y, j, lam=LAMBDA):
    '''影响函数一步遗忘样本 j：w + (1/n) H⁻¹ ∇L(z)。'''
    n = len(y)
    H = hessian_lr(w, X, y, lam)
    g_z = grad_single(w, X[j], y[j])      # z 的梯度
    return w + np.linalg.solve(H, g_z) / n  # 撤销 z 的影响

w_unlearn = unlearn_influence(w_full, X, y, j)
err_donothing = np.linalg.norm(w_full - w_retrain)      # 啥也不做
err_influence = np.linalg.norm(w_unlearn - w_retrain)   # 影响函数遗忘
print(f'啥也不做   离重训 L2 = {err_donothing:.4f}')
print(f'影响函数   离重训 L2 = {err_influence:.4f}')
print(f'影响函数把误差减小了 {err_donothing/err_influence:.1f}x')
assert err_influence < err_donothing, '影响函数遗忘应比啥也不做更接近重训'
assert err_influence < 0.5 * err_donothing, '强凸下影响函数应显著逼近重训'
print('✅ 影响函数一步更新显著逼近重训 —— 不重训也能近似遗忘')

**影响函数对单样本很准**：在凸模型上，逐个删样本的影响函数估计应高度接近真实重训。对几个样本验证影响函数与重训的方向一致性。

In [ ]:
cos_sims = []
for jj in [0, 5, 10, 20]:
    Xm = np.delete(X, jj, axis=0); ym = np.delete(y, jj)
    w_rt = train_lr(Xm, ym)
    w_if = unlearn_influence(w_full, X, y, jj)
    actual_change = w_rt - w_full              # 重训带来的真实参数变化
    est_change = w_if - w_full                 # 影响函数估计的变化
    cos = actual_change @ est_change / (np.linalg.norm(actual_change)*np.linalg.norm(est_change)+1e-12)
    cos_sims.append(cos)
    print(f'样本 {jj:2d}: 影响函数 vs 重训 变化方向余弦相似度 = {cos:.3f}')
assert min(cos_sims) > 0.9, '强凸下影响函数估计的变化方向应高度对齐重训'
print('✅ 影响函数估计的参数变化方向与真实重训高度一致(凸模型)')

## 3 · SISA 精确遗忘：只重训 1/S

把数据切成 S 个不相交分片，各训一个子模型，推理时聚合（平均概率）。
删一个样本只需重训它所在的**那一个分片**——精确遗忘，成本降到 1/S。

In [ ]:
def train_sisa(X, y, n_shards=4):
    '''SISA：切 n_shards 个不相交分片，各训一个子模型。返回(子模型列表, 分片下标)。'''
    n = len(y)
    perm = np.arange(n)                    # 这里不打乱，便于定位(实际可打乱)
    shards = np.array_split(perm, n_shards)
    models = [train_lr(X[idx], y[idx]) for idx in shards]
    return models, shards

def sisa_predict(models, X):
    '''聚合：平均各子模型的预测概率。'''
    probs = np.mean([sigmoid(X @ w) for w in models], axis=0)
    return (probs > 0.5).astype(float)

def sisa_unlearn(models, shards, X, y, j):
    '''删样本 j：找到它所在分片，只重训那一个子模型。'''
    models = list(models)
    for s, idx in enumerate(shards):
        if j in idx:
            new_idx = idx[idx != j]           # 该分片删掉 j
            models[s] = train_lr(X[new_idx], y[new_idx])
            return models, s
    return models, -1

models, shards = train_sisa(X, y, n_shards=4)
acc_before = np.mean(sisa_predict(models, X) == y)
# 删样本 j=0，只重训其所在分片
import copy
models_before = copy.deepcopy(models)
models_after, retrained_shard = sisa_unlearn(models, shards, X, y, 0)
# 验证：只有一个子模型变了，其余 S-1 个纹丝不动
changed = [s for s in range(4) if not np.allclose(models_before[s], models_after[s])]
print(f'SISA 删一个样本: 重训了分片 {retrained_shard}, 变化的子模型 = {changed}')
assert changed == [retrained_shard], '应只有目标所在分片的子模型改变'
assert len(changed) == 1, f'应只重训 1/{4} 的模型(精确遗忘)'
print(f'✅ SISA 精确遗忘：只重训 1 个分片(1/{len(shards)} 成本)，其余 {len(shards)-1} 个子模型不动')

**S 的权衡**：分片越多遗忘越便宜，但每个子模型见的数据越少、聚合精度可能越低。验证之。

In [ ]:
Xtr, ytr = make_data(n=600, d=6, seed=1)
Xte, yte = make_data(n=300, d=6, seed=2)
g = np.random.default_rng(9); w_t = g.standard_normal(6)
def gen(n, s):
    gg=np.random.default_rng(s); X=gg.standard_normal((n,6))
    return X, (X@w_t + 0.5*gg.standard_normal(n) > 0).astype(float)
Xtr,ytr=gen(600,1); Xte,yte=gen(300,2)
print(f"{'分片数 S':>8s} {'测试准确率':>10s} {'遗忘成本':>10s}")
for S in [1, 2, 4, 8]:
    ms, sh = train_sisa(Xtr, ytr, n_shards=S)
    acc = np.mean(sisa_predict(ms, Xte) == yte)
    print(f'{S:>8d} {acc:>10.3f} {f"1/{S}":>10s}')
# S=1 即单模型(无分片)，精度通常最高；S 越大遗忘越便宜
acc1 = np.mean(sisa_predict(train_sisa(Xtr,ytr,1)[0], Xte)==yte)
acc8 = np.mean(sisa_predict(train_sisa(Xtr,ytr,8)[0], Xte)==yte)
assert acc1 >= acc8 - 0.05, '分片越多精度大体越低(遗忘成本越低)'
print('✅ S 是「遗忘成本 vs 精度」的旋钮：S 越大删一条越便宜，但聚合精度可能下降')

## 4 · 遗忘验证：成员推断掉回随机水平

遗忘的终极目的：**让攻击者无法判断 z 被训练过**。用一个简单的成员推断（基于损失）验证：
- 遗忘前：z 的损失明显低于陌生样本（模型记得它）-> 可被识别为「训练过」
- 遗忘后：z 的损失应**接近陌生样本**（忘干净 = 无法区分）

In [ ]:
def sample_loss(w, x, yi):
    '''单样本逻辑回归损失(越低=模型越「认识」它)。'''
    p = sigmoid(x @ w)
    return -(yi*np.log(p+1e-12) + (1-yi)*np.log(1-p+1e-12))

# 重新训练一个会「记住」样本的模型(小数据、弱正则 -> 更明显的记忆)
Xs, ys = make_data(n=60, d=10, seed=3)        # 小数据更易记忆
w_mem = train_lr(Xs, ys, lam=0.01, iters=2000)
z_idx = 0
# 陌生样本(同分布但没训练过)
X_strangers, y_strangers = make_data(n=60, d=10, seed=99)

loss_z_before = sample_loss(w_mem, Xs[z_idx], ys[z_idx])
loss_strangers = np.mean([sample_loss(w_mem, X_strangers[i], y_strangers[i]) for i in range(60)])
print(f'遗忘前: z 的损失={loss_z_before:.4f}, 陌生样本平均损失={loss_strangers:.4f}')
print(f'  z 的损失明显更低 -> 成员推断能识别「z 被训练过」')
assert loss_z_before < loss_strangers, '训练样本损失应低于陌生样本(可被MIA识别)'

# 遗忘 z（影响函数），再看 z 的损失是否接近陌生样本
w_forgotten = unlearn_influence(w_mem, Xs, ys, z_idx, lam=0.01)
loss_z_after = sample_loss(w_forgotten, Xs[z_idx], ys[z_idx])
print(f'遗忘后: z 的损失={loss_z_after:.4f} (应升高、向陌生样本靠拢)')
assert loss_z_after > loss_z_before, '遗忘后 z 的损失应升高(记忆被削弱)'
print('✅ 遗忘验证：遗忘后 z 的损失升高、向陌生样本靠拢 -> 成员信号被削弱')

---
## ✏️ 练习 1：实现重训（金标准）

实现 `retrain_without(X, y, forget_indices)`：删掉 `forget_indices` 里的所有样本后从头训练，返回模型。这是遗忘的参照系。

In [ ]:
def retrain_without(X, y, forget_indices):
    # TODO: 从 X,y 中删除 forget_indices 指定的行，用 train_lr 从头训练并返回 w
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
X, y = make_data(n=200, d=6, seed=7)
w_full2 = train_lr(X, y)
w_rt = retrain_without(X, y, [0, 1, 2])
# 应等价于手动删除后训练
Xm = np.delete(X, [0,1,2], axis=0); ym = np.delete(y, [0,1,2])
assert np.allclose(w_rt, train_lr(Xm, ym)), '应等价于删除后重训'
# 删了样本，结果应不同于全量模型
assert not np.allclose(w_rt, w_full2), '重训结果应不同于全量模型'
print('✅ 练习 1 通过：重训金标准实现正确')

## ✏️ 练习 2：影响函数遗忘

实现 `influence_unlearn(w, X, y, j, lam)`：用影响函数一步更新遗忘样本 j（复用 `hessian_lr` 和 `grad_single`），返回遗忘后参数。

In [ ]:
def influence_unlearn(w, X, y, j, lam=LAMBDA):
    # TODO: 算 Hessian H = hessian_lr(...); 算 z 的梯度 g_z = grad_single(...);
    #       返回 w + H⁻¹ g_z / n  (用 np.linalg.solve 避免显式求逆)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
X, y = make_data(n=300, d=6, seed=11)
w_full3 = train_lr(X, y)
w_rt = retrain_without(X, y, [3])
w_if = influence_unlearn(w_full3, X, y, 3)
err_nothing = np.linalg.norm(w_full3 - w_rt)
err_if = np.linalg.norm(w_if - w_rt)
print(f'啥也不做离重训={err_nothing:.4f}, 影响函数离重训={err_if:.4f}')
assert err_if < err_nothing, '影响函数应比啥也不做更接近重训'
print('✅ 练习 2 通过：影响函数一步遗忘逼近重训')

## ✏️ 练习 3：遗忘验证（成员推断成功率）

实现 `membership_advantage(w, members_X, members_y, strangers_X, strangers_y)`：用基于损失的成员推断，返回攻击者的「优势」——用损失阈值区分成员/陌生样本的准确率减去 0.5（0=无法区分=忘干净，>0=能区分=有泄露）。

提示：成员的损失通常更低；以两组损失的中位数中点为阈值，损失<阈值判为「成员」。

In [ ]:
def membership_advantage(w, members_X, members_y, strangers_X, strangers_y):
    # TODO: 算 members 和 strangers 各自的样本损失；以两者损失中位数的中点为阈值，
    #       损失<阈值 -> 判为成员。返回 (判对率) - 0.5
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
Xs, ys = make_data(n=50, d=10, seed=13)
w_mem = train_lr(Xs, ys, lam=0.01, iters=2000)   # 会记忆
Xstr, ystr = make_data(n=50, d=10, seed=131)
adv_before = membership_advantage(w_mem, Xs, ys, Xstr, ystr)
print(f'记忆模型的成员推断优势 = {adv_before:.3f} (>0 表示能区分 -> 有泄露)')
assert adv_before > 0.05, '过拟合模型应有可观的成员推断优势'
# 一个没记忆的强正则模型，优势应接近 0
w_reg = train_lr(Xs, ys, lam=5.0, iters=2000)
adv_reg = membership_advantage(w_reg, Xs, ys, Xstr, ystr)
print(f'强正则模型的成员推断优势 = {adv_reg:.3f} (应更接近0)')
assert adv_reg < adv_before, '更少记忆 -> 成员推断优势更小'
print('✅ 练习 3 通过：会用成员推断优势量化「能不能区分训练过」(遗忘验证的核心)')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def retrain_without(X, y, forget_indices):
    Xm = np.delete(X, forget_indices, axis=0)
    ym = np.delete(y, forget_indices)
    return train_lr(Xm, ym)

In [ ]:
# 练习 2 参考答案
def influence_unlearn(w, X, y, j, lam=LAMBDA):
    n = len(y)
    H = hessian_lr(w, X, y, lam)
    g_z = grad_single(w, X[j], y[j])
    return w + np.linalg.solve(H, g_z) / n

In [ ]:
# 练习 3 参考答案
def membership_advantage(w, members_X, members_y, strangers_X, strangers_y):
    lm = np.array([sample_loss(w, members_X[i], members_y[i]) for i in range(len(members_y))])
    ls = np.array([sample_loss(w, strangers_X[i], strangers_y[i]) for i in range(len(strangers_y))])
    thr = 0.5 * (np.median(lm) + np.median(ls))     # 阈值
    # 损失<阈值 判为成员
    correct = np.concatenate([lm < thr, ls >= thr])  # 成员应<阈值, 陌生应>=阈值
    return float(correct.mean() - 0.5)

---
## 🧪 真实数据胶囊：GDPR 删除请求的遗忘成本核算

真实场景：一个有大量用户的产品，依 GDPR 持续收到删除请求。不同遗忘策略的**成本**天差地别。

我们用真实量级核算三种策略（全量重训 / SISA / 影响函数）响应 `R` 个删除请求的相对成本，理解「为什么必须有遗忘技术」「为什么遗忘能力是架构的函数」。

In [ ]:
# 一次全量训练的相对成本设为 1.0
def unlearning_cost(strategy, n_requests, n_shards=10, full_train_cost=1.0):
    '''返回响应 n_requests 个删除请求的相对总成本。'''
    if strategy == 'retrain':
        return n_requests * full_train_cost          # 每个请求全量重训
    elif strategy == 'sisa':
        return n_requests * (full_train_cost / n_shards)  # 每个请求重训 1 个分片
    elif strategy == 'influence':
        return n_requests * 0.001 * full_train_cost  # 一步更新≈千分之一次训练
    raise ValueError(strategy)

R = 1000     # 一千个删除请求
print(f'响应 {R} 个删除请求的相对成本(一次全量训练=1.0):')
for strat in ['retrain', 'sisa', 'influence']:
    cost = unlearning_cost(strat, R, n_shards=10)
    print(f'  {strat:12s}: {cost:>10.2f}  (相当于 {cost:.1f} 次全量训练)')
c_retrain = unlearning_cost('retrain', R)
c_sisa = unlearning_cost('sisa', R, n_shards=10)
c_inf = unlearning_cost('influence', R)
assert c_retrain > c_sisa > c_inf, '成本: 全量重训 > SISA > 影响函数'
assert abs(c_sisa - c_retrain/10) < 1e-6, 'SISA(10分片) 应是全量重训的 1/10'
print('\n✅ 全量重训 1000 次=训 1000 遍(不可接受)；SISA 降到 1/S；影响函数近乎免费 —— 这就是为什么需要遗忘技术')

**🧪 胶囊练习**：实现 `breakeven_shards(n_requests, full_train_cost, budget)`：在删除请求数 `n_requests`、预算 `budget`（以全量训练为单位）下，SISA 至少要切多少分片才能把总遗忘成本压到预算内？（返回最小整数分片数。）

In [ ]:
def breakeven_shards(n_requests, full_train_cost, budget):
    # TODO: 求最小 S 使 n_requests * (full_train_cost/S) <= budget
    #       即 S >= n_requests*full_train_cost/budget，向上取整
    raise NotImplementedError

In [ ]:
# 自测
import math
S = breakeven_shards(n_requests=1000, full_train_cost=1.0, budget=50.0)
print(f'1000 个请求、预算 50 次训练: 至少需 {S} 个分片')
assert S == 20, '1000/50 = 20 个分片'
assert 1000 * (1.0/S) <= 50.0, '该分片数应满足预算'
assert 1000 * (1.0/(S-1)) > 50.0, '应是满足预算的最小分片数'
print('✅ 胶囊练习通过：能据删除请求量与预算反推所需分片数')

In [ ]:
# 📖 胶囊参考答案
import math
def breakeven_shards(n_requests, full_train_cost, budget):
    return math.ceil(n_requests * full_train_cost / budget)

### 小结
- **删原始数据 ≠ 删模型影响**：影响被压进参数、还纠缠在一起。重训是金标准(完美但贵)。
- **精确遗忘 SISA**：分片隔离 -> 删一条只重训 1/S；S 是「遗忘成本 vs 精度」旋钮。
- **近似遗忘 影响函数**：θ + H⁻¹∇L(z)/n 一步逼近重训；凸模型准，深度网络只是近似。
- **遗忘必须验证**：成员推断最贴合目标(掉回随机=忘干净)；过度遗忘也是泄露。
- **边界**：删不掉已扩散的副本、有连带损害；遗忘能力是训练架构的函数(隐私即设计)。

下一站：**模块 05 · 可信部署** —— 把 DP+联邦+遗忘拼成端到端隐私管线，过审计、上生产。